# Kazakh Punctuation Prediction Framework

**Task:** Given a sequence of lowercase, punctuation-stripped Kazakh words, predict a punctuation label for each word.

**Data sources:** [kz-transformers/multidomain-kazakh-dataset](https://huggingface.co/datasets/kz-transformers/multidomain-kazakh-dataset)

Runnable on Google Colab / Kaggle.

## 1. Setup & Install

In [4]:
# Install dependencies (run once)
# Data: datasets, pandas, huggingface_hub, pyarrow, regex, tqdm
# ML: torch, transformers (install separately if training)
!pip3 install -q datasets pandas huggingface_hub pyarrow regex tqdm transformers torch

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
import os
import re
import pandas as pd
from pathlib import Path
from typing import List, Tuple, Optional
from tqdm import tqdm

from datasets import load_dataset

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Config: paths & options
OUTPUT_DIR = Path("./kazakh_punct_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# Dataset: 5 CSV files from HuggingFace
DATASET_ID = "kz-transformers/multidomain-kazakh-dataset"
HF_CSV_FILES = [
    "cc100-monolingual-crawled-data.csv",
    "kazakhBooks.csv",
    "kazakhNews.csv",
    "leipzig.csv",
    "oscar.csv",
]

# Labels (from train_example.csv)
LABELS = ["O", "COMMA", "PERIOD", "QUESTION"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

# Quick test: load only leipzig.csv (389MB) for fast Colab iteration
QUICK_TEST = True  # Set False to load all 5 splits
# Sampling (set to None to use full data; use for Colab RAM limits)
MAX_TEXTS_PER_SPLIT = 10_000 if QUICK_TEST else 50_000  # Sample per CSV to avoid OOM
# MAX_TEXTS_PER_SPLIT = None  # Uncomment for full data (requires more RAM)

## 2. Load All Datasets

In [ ]:
def load_csv_from_hf(split_name: str, max_rows: Optional[int] = None) -> pd.DataFrame:
    """Load a single CSV from HuggingFace dataset (uses resolve URL)."""
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/main/{split_name}"
    # Use low_memory=False for consistent dtypes; on_bad_lines='skip' for malformed rows
    df = pd.read_csv(url, nrows=max_rows, encoding="utf-8", on_bad_lines="skip")
    df["source"] = split_name.replace(".csv", "")
    return df


def load_all_splits(max_per_split: Optional[int] = None, files_to_load: Optional[List[str]] = None) -> pd.DataFrame:
    """Load CSV splits from multidomain Kazakh dataset."""
    file_list = files_to_load if files_to_load else HF_CSV_FILES
    all_dfs = []
    for fname in tqdm(file_list, desc="Loading splits"):
        try:
            df = load_csv_from_hf(fname, max_rows=max_per_split)
            all_dfs.append(df)
            print(f"  {fname}: {len(df):,} rows")
        except Exception as e:
            print(f"  {fname}: FAILED - {e}")
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

In [ ]:
# Load datasets (QUICK_TEST=only leipzig; otherwise all 5 splits)
files = ["leipzig.csv"] if QUICK_TEST else None
print("Loading multidomain Kazakh dataset...")
raw_df = load_all_splits(max_per_split=MAX_TEXTS_PER_SPLIT, files_to_load=files)
print(f"\nTotal texts loaded: {len(raw_df):,}")
print(raw_df.head(2))

## 3. Preprocessing

In [ ]:
# Punctuation mapping to labels (from train_example.csv)
PUNCT_TO_LABEL = {
    ",": "COMMA",
    ".": "PERIOD",
    "?": "QUESTION",
    "!": "PERIOD",   # map exclamation to period
    ";": "COMMA",    # clause boundary
    ":": "COMMA",
    "—": "COMMA",    # em dash
    "–": "COMMA",    # en dash
}

# Quote marks to strip from words (don't predict; Kazakh uses « »)
QUOTE_CHARS = set('"\'«»\u2018\u2019\u201c\u201d')

In [ ]:
def extract_trailing_punct(token: str) -> Tuple[str, str]:
    """Extract trailing punctuation from token. Returns (clean_word, punct)."""
    if not token:
        return "", ""
    punct = ""
    end_idx = len(token)
    for i in range(len(token) - 1, -1, -1):
        c = token[i]
        if c in PUNCT_TO_LABEL or c in QUOTE_CHARS:
            punct = c + punct
            end_idx = i
        else:
            break
    word = token[:end_idx]
    # Strip leading quotes from word
    while word and word[0] in QUOTE_CHARS:
        word = word[1:]
    return word, punct


def punct_to_label(p: str) -> str:
    """Map punctuation char to label."""
    if not p:
        return "O"
    # Take first char that we predict
    for c in p:
        if c in PUNCT_TO_LABEL:
            return PUNCT_TO_LABEL[c]
    return "O"

In [ ]:
def text_to_punct_examples(text: str) -> List[Tuple[str, str]]:
    """
    Convert raw text into (input_text, labels) format matching train_example.csv.
    Each example is one sentence: lowercase, punctuation-stripped words + space-separated labels.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    
    text = text.strip()
    # Split into sentences by sentence-ending punctuation
    # Keep . ? ! as delimiters but also capture them for the previous segment
    sent_pattern = re.split(r'(?<=[.!?])\s+', text)
    examples = []
    
    for sent in sent_pattern:
        sent = sent.strip()
        if not sent:
            continue
        
        tokens = sent.split()
        words = []
        labels = []
        
        for tok in tokens:
            word, trailing = extract_trailing_punct(tok)
            # Skip empty tokens
            if not word:
                continue
            word_lower = word.lower()
            words.append(word_lower)
            lbl = punct_to_label(trailing)
            labels.append(lbl)
        
        if words and labels:
            # Ensure last token has PERIOD/QUESTION if sentence ended with . or ?
            # (already captured in extract_trailing_punct)
            input_text = " ".join(words)
            label_str = " ".join(labels)
            # Filter very short or noisy segments
            if len(words) >= 2 and len(input_text) > 3:
                examples.append((input_text, label_str))
    
    return examples

In [ ]:
def preprocess_raw_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocess raw dataframe: extract text, filter Kazakh, convert to train format.
    Returns dataframe with columns: id, input_text, labels, source
    """
    # Use 'text' column; fallback to first col if missing
    text_col = "text" if "text" in df.columns else df.columns[0]
    
    # Optional: filter by language if column exists
    if "predicted_language" in df.columns:
        df = df[df["predicted_language"] == "kaz"].copy()
    
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing"):
        text = row[text_col]
        if pd.isna(text) or not str(text).strip():
            continue
        source = row.get("source", "unknown")
        examples = text_to_punct_examples(text)
        for i, (inp, lbl) in enumerate(examples):
            ex_id = f"punct_{source}_{idx}_{i}"
            rows.append({"id": ex_id, "input_text": inp, "labels": lbl, "source": source})
    
    return pd.DataFrame(rows)

In [ ]:
# Run preprocessing
print("Preprocessing raw text to train format...")
processed_df = preprocess_raw_df(raw_df)
print(f"\nProcessed examples: {len(processed_df):,}")
print(processed_df.head(5))

## 4. Combine with Hackathon Train Example (Optional)

In [ ]:
# Path to hackathon train example (if in same repo)
TRAIN_EXAMPLE_PATH = "kaz-punct-hackathon/train_example.csv"  # adjust path in Colab/Kaggle

def load_hackathon_train(path: str) -> pd.DataFrame:
    """Load the provided train_example.csv."""
    try:
        df = pd.read_csv(path)
        df["source"] = "hackathon_train"
        return df
    except FileNotFoundError:
        print(f"Train example not found at {path}. Skipping merge.")
        return pd.DataFrame()

# Merge if available
if os.path.exists(TRAIN_EXAMPLE_PATH):
    hackathon_df = load_hackathon_train(TRAIN_EXAMPLE_PATH)
    processed_df = pd.concat([hackathon_df, processed_df], ignore_index=True)
    print(f"Merged. Total examples: {len(processed_df):,}")
else:
    print("Train example not found. Using only preprocessed HF data.")

## 5. Store for Downstream ML

In [ ]:
def save_for_ml(df: pd.DataFrame, out_dir: Path) -> dict:
    """Save processed data in formats suitable for ML (train/val split, parquet, csv)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Shuffle and split (90% train, 10% val)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(df)
    split_idx = int(0.9 * n)
    train_df = df.iloc[:split_idx]
    val_df = df.iloc[split_idx:]
    
    # Save full, train, val
    full_path = out_dir / "processed_full.parquet"
    train_path = out_dir / "train.parquet"
    val_path = out_dir / "val.parquet"
    csv_path = out_dir / "train_labels.csv"  # CSV for easy inspection
    
    df.to_parquet(full_path, index=False)
    train_df.to_parquet(train_path, index=False)
    val_df.to_parquet(val_path, index=False)
    train_df[["id", "input_text", "labels"]].to_csv(csv_path, index=False)
    
    # Save label vocab
    with open(out_dir / "labels.txt", "w") as f:
        f.write("\n".join(LABELS))
    
    return {
        "full": str(full_path),
        "train": str(train_path),
        "val": str(val_path),
        "train_csv": str(csv_path),
        "train_count": len(train_df),
        "val_count": len(val_df),
    }

In [ ]:
# Save and report
paths = {}
if len(processed_df) > 0:
    paths = save_for_ml(processed_df, OUTPUT_DIR)
    print("Saved:")
    for k, v in paths.items():
        print(f"  {k}: {v}")
    sample = pd.read_csv(paths["train_csv"], nrows=3)
    print("\nSample (matches train_example.csv format):")
    print(sample)
else:
    print("Skipped save: processed_df is empty.")

## 6. PyTorch Dataset (for ML training)

In [ ]:
# Optional: PyTorch Dataset for tokenization + training
# Uncomment and pip install transformers torch if needed

# from torch.utils.data import Dataset
#
# class KazakhPunctDataset(Dataset):
#     def __init__(self, df, tokenizer, max_length=128):
#         self.df = df
#         self.tokenizer = tokenizer
#         self.max_length = max_length
#         self.label2id = LABEL2ID
#
#     def __len__(self):
#         return len(self.df)
#
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         words = row["input_text"].split()
#         labels = row["labels"].split()
#         # Tokenize word-by-word, align labels...
#         return {"input_ids": ..., "labels": ...}

## Summary

In [ ]:
print("Framework complete.")
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Labels: {LABELS}")
if processed_df is not None and len(processed_df) > 0:
    print(f"Train examples: {paths.get('train_count', 'N/A')}")
    print(f"Val examples: {paths.get('val_count', 'N/A')}")
else:
    print("No data saved (processed_df was empty).")

## 7. ML Training (Punctuation Restoration)

Run training with XLM-RoBERTa-base (default) or other models. Models: `xlm-roberta-base`, `xlm-roberta-large`, `kaz-roberta`, `kazbert`, `bert-base-kazakh`, `bert-base-multilingual-cased`.

Metrics: accuracy, precision, recall, macro F1 (excluding class O), F1. Best model saved by macro F1 (excl O).

In [ ]:
# Models: kaz-roberta (Kazakh), kazbert, xlm-roberta-base, xlm-roberta-large
# New: 2-layer BiLSTM, MLP classifier, residual connection, seq_len=256
!python -m src.train --model kaz-roberta --data-path ./kazakh_punct_data --save-path ./out --epochs 15 --batch-size 16 --sequence-length 256

## 8. Evaluate on test.parquet (before submission)

Run the trained model on your labeled test split to get metrics (accuracy, macro F1, etc.) before submitting to the hackathon.

In [ ]:
# Evaluate on test split before submission (test.parquet)
from src.inference import evaluate_on_test

TEST_PARQUET = "kazakh_punct_data/test.parquet"
WEIGHT_PATH = "./out/best.pt"

test_metrics = evaluate_on_test(
    test_path=TEST_PARQUET,
    weight_path=WEIGHT_PATH,
)
print("\nProceeding to submission inference...")

## 9. Run Test (Submission)

Load `best.pt` and run inference on `kaz-punct-hackathon/test.csv`. Outputs submission CSV with `id,labels`.

In [ ]:
# Config: must match model used for training
INFERENCE_MODEL = "xlm-roberta-base"  # same as --model in training
WEIGHT_PATH = "./out/best.pt"
TEST_CSV = "kaz-punct-hackathon/test.csv"
SUBMISSION_CSV = "kaz-punct-hackathon/submission.csv"

from src.inference import run_inference

submission_df = run_inference(
    model_name=INFERENCE_MODEL,
    weight_path=WEIGHT_PATH,
    test_csv=TEST_CSV,
    sequence_len=256,
)
submission_df.to_csv(SUBMISSION_CSV, index=False)
print(f"Saved {len(submission_df)} predictions to {SUBMISSION_CSV}")
print(submission_df.head())